# Chapter 17: Limited Dependent Variable Models and Sample Selection Corrections

Topics: binary responses (LPM, Logit, Probit), count data (Poisson), Tobit model, censored/truncated regression, sample selection.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('mroz')

## 17.1.1 Linear Probability Model

In [ ]:
lpm = smf.ols('inlf ~ nwifeinc + educ + exper + I(exper**2) + age + kidslt6 + kidsge6',
               data=df).fit()
print("Linear Probability Model:")
print(lpm.summary())
print(f"\nWarning: {(lpm.fittedvalues < 0).sum()} predictions < 0, "
      f"{(lpm.fittedvalues > 1).sum()} predictions > 1")

## 17.1.2 Logit and Probit

In [ ]:
logit  = smf.logit('inlf ~ nwifeinc + educ + exper + I(exper**2) + age + kidslt6',
                    data=df).fit()
probit = smf.probit('inlf ~ nwifeinc + educ + exper + I(exper**2) + age + kidslt6',
                     data=df).fit()
print("Logit Results:")
print(logit.summary())

## 17.1.5 Marginal Effects

In [ ]:
mfx_logit  = logit.get_margeff()
mfx_probit = probit.get_margeff()

comp = pd.DataFrame({
    'LPM':    lpm.params.drop('Intercept'),
    'Logit AME':  mfx_logit.margeff,
    'Probit AME': mfx_probit.margeff
})
print("Average Marginal Effects Comparison:")
print(comp.round(4))

## 17.2 Count Data: Poisson Regression

In [ ]:
df_fert = woo.dataWoo('fertil1')
poisson = smf.poisson('kids ~ educ + age + agesq + black + east + northcen + west + farm',
                       data=df_fert).fit()
print("Poisson Results:")
print(poisson.summary())

# Incidence rate ratios
irr = np.exp(poisson.params)
print("\nIncidence Rate Ratios:")
print(irr.round(4))

## 17.3 Tobit Model

In [ ]:
# Hours worked (censored at 0 for non-participants)
# Standard Tobit via statsmodels (OLS on uncensored subsample as approximation)
tobit_ols = smf.ols('hours ~ nwifeinc + educ + exper + I(exper**2) + age + kidslt6',
                     data=df[df['inlf']==1]).fit()
print("OLS on uncensored subsample (upper bound of Tobit):")
print(tobit_ols.summary())